In [1]:
!pip install nilearn nibabel scikit-learn seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 106.2 MB/s eta 0:00:00


In [3]:
import os
import random
import copy
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models.video as models_video
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from nilearn import datasets
import matplotlib.pyplot as plt
import seaborn as sns
import gc

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


In [4]:
abide = datasets.fetch_abide_pcp(n_subjects=150, pipeline='cpac', data_dir='/content/abide_data')

fmri_files = abide.func_preproc
labels = abide.phenotypic['DX_GROUP']
# DX_GROUP: 1 = Аутизм, 2 = Контроль. Переводим в 0 и 1
labels = [1 if label == 1 else 0 for label in labels]

print(f"Скачано файлов: {len(fmri_files)}")
print(f"Классы: Контроль (0) = {labels.count(0)}, Аутизм (1) = {labels.count(1)}")

train_files, val_files, train_labels, val_labels = train_test_split(
    fmri_files, labels, test_size=0.2, random_state=SEED, stratify=labels
)
print(f"Обучение: {len(train_files)} | Валидация: {len(val_files)}")

[fetch_abide_pcp] Added README.md to /content/abide_data

[fetch_abide_pcp] Dataset created in /content/abide_data/ABIDE_pcp

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv ...

[fetch_abide_pcp]  ...done. (0 seconds, 0 min)

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/func_preproc/Pitt_005
0003_func_preproc.nii.gz ...

...

[fetch_abide_pcp]  ...done. (2 seconds, 0 min)

[fetch_abide_pcp] Downloading data from 
https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/nofilt_noglobal/func_preproc/Trinity_
0050257_func_preproc.nii.gz ...

[fetch_abide_pcp] Downloaded 53141504 of 86334444 bytes (61.6%%,    0.6s remaining)

[fetch_abide_pcp]  ...done. (2 seconds, 0 min)

Скачано файлов: 150
Классы: Контроль (0) = 73, Аутизм (1) = 77
Обучение: 120 | Валидация: 30


In [5]:
# ==========================================
# 0. INSTALL
# ==========================================
!pip -q install nilearn nibabel

# ==========================================
# 1. IMPORTS
# ==========================================
import os
import gc
import copy
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import nibabel as nib
from nilearn import datasets

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models.video as models_video
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")

# ==========================================
# 2. CONFIG
# ==========================================
@dataclass
class CFG:
    seed: int = 42
    data_dir: str = "/content/abide_data"
    n_subjects: int = 150
    pipeline: str = "cpac"
    derivative: str = "func_preproc"

    seq_length: int = 30
    batch_size: int = 2
    num_workers: int = 2
    pin_memory: bool = True

    epochs: int = 15
    lr: float = 1e-4
    weight_decay: float = 1e-3
    grad_clip: float = 1.0

    # AES / Adaptive DIBL params
    max_alpha: float = 0.05
    compression_ratio: float = 0.25
    warmup_epochs: int = 4

    # split
    val_size: float = 0.2

CFG = CFG()

# ==========================================
# 3. REPRODUCIBILITY
# ==========================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================================
# 4. DATA DOWNLOAD
# ==========================================
print("Downloading ABIDE...")
abide = datasets.fetch_abide_pcp(
    data_dir=CFG.data_dir,
    n_subjects=CFG.n_subjects,
    pipeline=CFG.pipeline,
    derivatives=[CFG.derivative],
)

fmri_files = abide.func_preproc
labels = [1 if dx == 1 else 0 for dx in abide.phenotypic["DX_GROUP"]]

print(f"Downloaded files: {len(fmri_files)}")
print(f"ASD: {sum(labels)} | Control: {len(labels) - sum(labels)}")

train_files, val_files, train_labels, val_labels = train_test_split(
    fmri_files,
    labels,
    test_size=CFG.val_size,
    random_state=CFG.seed,
    stratify=labels,
)

print(f"Train: {len(train_files)} | Val: {len(val_files)}")

# ==========================================
# 5. DATASET
# ==========================================
class ABIDEDataset(Dataset):
    """
    train  -> random temporal crop
    val    -> central temporal crop
    z-score normalization per subject
    output -> [T, 1, H, W, D]
    """
    def __init__(self, file_paths, labels, seq_length=30, mode="train"):
        self.file_paths = file_paths
        self.labels = labels
        self.seq_length = seq_length
        self.mode = mode

    def __len__(self):
        return len(self.file_paths)

    def _crop_or_pad(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [T, H, W, D]
        T_total = x.shape[0]

        if T_total >= self.seq_length:
            if self.mode == "train":
                start_t = random.randint(0, T_total - self.seq_length)
            else:
                start_t = (T_total - self.seq_length) // 2
            x = x[start_t:start_t + self.seq_length]
        else:
            pad_len = self.seq_length - T_total
            pad = torch.zeros((pad_len, *x.shape[1:]), dtype=x.dtype)
            x = torch.cat([x, pad], dim=0)

        return x

    def __getitem__(self, idx):
        data = nib.load(self.file_paths[idx]).get_fdata()   # [H, W, D, T]
        x = torch.tensor(data, dtype=torch.float32).permute(3, 0, 1, 2)  # [T, H, W, D]
        x = self._crop_or_pad(x)

        # per-subject z-score
        x = (x - x.mean()) / (x.std() + 1e-8)

        # [T, 1, H, W, D]
        x = x.unsqueeze(1)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_ds = ABIDEDataset(train_files, train_labels, seq_length=CFG.seq_length, mode="train")
val_ds   = ABIDEDataset(val_files,   val_labels,   seq_length=CFG.seq_length, mode="val")

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
)

# ==========================================
# 6. MODEL
# ==========================================
class Robust3DResNetLSTM(nn.Module):
    def __init__(self, lstm_hidden=256, num_classes=2):
        super().__init__()
        self.encoder = models_video.r3d_18(weights=None)

        # adapt to 1-channel fMRI
        orig_stem = self.encoder.stem[0]
        self.encoder.stem[0] = nn.Conv3d(
            1,
            orig_stem.out_channels,
            kernel_size=orig_stem.kernel_size,
            stride=orig_stem.stride,
            padding=orig_stem.padding,
            bias=False,
        )
        self.encoder.fc = nn.Identity()  # output 512

        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: [B, T, 1, H, W, D]
        B, T, C, H, W, D = x.shape
        x = x.view(B * T, C, H, W, D)
        features = self.encoder(x)       # [B*T, 512]
        features = features.view(B, T, -1)
        _, (h_n, _) = self.lstm(features)
        logits = self.classifier(h_n[-1])
        return logits

# ==========================================
# 7. AES LOSS
# ==========================================
class AdaptiveDynamicIBLoss(nn.Module):
    """
    CrossEntropy + one-sided entropy barrier on deep activations.
    early_activations: reference layer activations
    deep_activations : compressed layer activations
    """
    def __init__(self, max_alpha=0.05, compression_ratio=0.25):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.max_alpha = max_alpha
        self.compression_ratio = compression_ratio

    @staticmethod
    def _entropy_from_activations(act: torch.Tensor) -> torch.Tensor:
        # act: [B, F]
        sigma = F.softmax(act, dim=-1)
        entropy = -(sigma * torch.log2(sigma + 1e-8)).sum(dim=-1).mean()
        return entropy

    def forward(
        self,
        logits,
        targets,
        early_activations,
        deep_activations,
        current_epoch,
        warmup_epochs=4,
    ):
        base_loss = self.ce_loss(logits, targets)

        with torch.no_grad():
            sigma_0 = F.softmax(early_activations[0], dim=-1)
            initial_entropy = -(sigma_0 * torch.log2(sigma_0 + 1e-8)).sum(dim=-1).mean()
            dynamic_target = initial_entropy * self.compression_ratio

        if current_epoch < warmup_epochs:
            current_alpha = self.max_alpha * (current_epoch + 1) / warmup_epochs
        else:
            current_alpha = self.max_alpha

        entropy_penalty = 0.0
        for act in deep_activations:
            entropy = self._entropy_from_activations(act)
            entropy_penalty = entropy_penalty + torch.relu(entropy - dynamic_target)

        total_loss = base_loss + current_alpha * entropy_penalty
        return total_loss

# ==========================================
# 8. METRIC HELPERS
# ==========================================
def compute_entropy_from_flat_acts(flat_acts: torch.Tensor) -> float:
    sigma = F.softmax(flat_acts, dim=-1)
    entropy = -(sigma * torch.log2(sigma + 1e-8)).sum(dim=-1).mean()
    return float(entropy.item())

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_targets = []
    all_probs = []
    all_preds = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_targets.extend(y.numpy().tolist())

    acc = accuracy_score(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)
    return {
        "accuracy": acc,
        "roc_auc": auc,
        "targets": np.array(all_targets),
        "probs": np.array(all_probs),
        "preds": np.array(all_preds),
    }

@torch.no_grad()
def compute_hidden_entropy(model, loader, hook_layer_name="layer4"):
    """
    Mean entropy of deep hidden space on validation set.
    """
    model.eval()
    deep_acts = []

    def _hook(module, inp, out):
        deep_acts.append(out.view(out.shape[0], -1))

    layer = getattr(model.encoder, hook_layer_name)
    handle = layer.register_forward_hook(_hook)

    entropies = []
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        deep_acts.clear()
        _ = model(x)
        if len(deep_acts) > 0:
            entropies.append(compute_entropy_from_flat_acts(deep_acts[0]))

    handle.remove()
    return float(np.mean(entropies)) if entropies else float("nan")

# ==========================================
# 9. TRAINING
# ==========================================
def train_model(use_aes: bool):
    model = Robust3DResNetLSTM().to(device)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay,
    )
    aes_criterion = AdaptiveDynamicIBLoss(
        max_alpha=CFG.max_alpha,
        compression_ratio=CFG.compression_ratio,
    )

    best_state = None
    best_metrics = None
    best_score = -np.inf

    early_acts, deep_acts = [], []

    def get_early(module, inp, out):
        early_acts.append(out.view(out.shape[0], -1))

    def get_deep(module, inp, out):
        deep_acts.append(out.view(out.shape[0], -1))

    if use_aes:
        h1 = model.encoder.layer1.register_forward_hook(get_early)
        h2 = model.encoder.layer4.register_forward_hook(get_deep)
    else:
        h1 = h2 = None

    for epoch in range(CFG.epochs):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if use_aes:
                early_acts.clear()
                deep_acts.clear()

            logits = model(x)

            if use_aes:
                loss = aes_criterion(
                    logits=logits,
                    targets=y,
                    early_activations=early_acts,
                    deep_activations=deep_acts,
                    current_epoch=epoch,
                    warmup_epochs=CFG.warmup_epochs,
                )
            else:
                loss = F.cross_entropy(logits, y)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            optimizer.step()

            running_loss += loss.item()

        val_metrics = evaluate_model(model, val_loader)

        print(
            f"[{'AES' if use_aes else 'BASE'}] "
            f"Epoch {epoch+1:02d}/{CFG.epochs} | "
            f"train_loss={running_loss/len(train_loader):.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_auc={val_metrics['roc_auc']:.4f}"
        )

        # keep best by validation accuracy; if tie, use AUC
        score = (val_metrics["accuracy"], val_metrics["roc_auc"])
        if score > best_score if isinstance(best_score, tuple) else True:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = val_metrics

    if use_aes:
        h1.remove()
        h2.remove()

    model.load_state_dict(best_state)

    hidden_entropy = compute_hidden_entropy(model, val_loader, hook_layer_name="layer4")
    best_metrics["hidden_entropy_bits"] = hidden_entropy

    del optimizer
    gc.collect()
    torch.cuda.empty_cache()

    return model, best_metrics

# ==========================================
# 10. RUN BASELINE
# ==========================================
baseline_model, baseline_metrics = train_model(use_aes=False)
print("\nBaseline metrics:")
print(baseline_metrics)

# ==========================================
# 11. RUN AES
# ==========================================
aes_model, aes_metrics = train_model(use_aes=True)
print("\nAES metrics:")
print(aes_metrics)

# ==========================================
# 12. FINAL TABLE
# ==========================================
baseline_entropy = baseline_metrics["hidden_entropy_bits"]
aes_entropy = aes_metrics["hidden_entropy_bits"]
compression_reduction = 100.0 * (1.0 - (aes_entropy / baseline_entropy))

print("\n" + "=" * 70)
print("FINAL RESULTS FOR ARTICLE")
print("=" * 70)
print(f"Baseline accuracy:        {baseline_metrics['accuracy'] * 100:.1f}")
print(f"Baseline ROC-AUC:         {baseline_metrics['roc_auc']:.3f}")
print(f"Baseline hidden entropy:  {baseline_entropy:.2f} bit")

print("-" * 70)

print(f"AES accuracy:             {aes_metrics['accuracy'] * 100:.1f}")
print(f"AES ROC-AUC:              {aes_metrics['roc_auc']:.3f}")
print(f"AES hidden entropy:       {aes_entropy:.2f} bit")
print(f"Redundancy reduction:     {compression_reduction:.1f}%")

# Convenient dict for copy-paste
article_table = {
    "Baseline": {
        "accuracy_percent": round(baseline_metrics["accuracy"] * 100, 1),
        "roc_auc": round(baseline_metrics["roc_auc"], 3),
        "hidden_entropy_bits": round(baseline_entropy, 2),
        "reduction_percent": "—",
    },
    "AES": {
        "accuracy_percent": round(aes_metrics["accuracy"] * 100, 1),
        "roc_auc": round(aes_metrics["roc_auc"], 3),
        "hidden_entropy_bits": round(aes_entropy, 2),
        "reduction_percent": round(compression_reduction, 1),
    },
}

print("\nCopy-paste table dict:")
print(article_table)

Device: cuda


[fetch_abide_pcp] Dataset found in /content/abide_data/ABIDE_pcp

Downloaded files: 150
ASD: 77 | Control: 73
Train: 120 | Val: 30
[BASE] Epoch 01/15 | train_loss=0.6999 | val_acc=0.5000 | val_auc=0.4444
[BASE] Epoch 02/15 | train_loss=0.6921 | val_acc=0.5000 | val_auc=0.6400
[BASE] Epoch 03/15 | train_loss=0.6895 | val_acc=0.5000 | val_auc=0.5156
[BASE] Epoch 04/15 | train_loss=0.7054 | val_acc=0.5000 | val_auc=0.6444
[BASE] Epoch 05/15 | train_loss=0.6928 | val_acc=0.5000 | val_auc=0.7067
[BASE] Epoch 06/15 | train_loss=0.6964 | val_acc=0.5000 | val_auc=0.7911
[BASE] Epoch 07/15 | train_loss=0.6916 | val_acc=0.5000 | val_auc=0.7244
[BASE] Epoch 08/15 | train_loss=0.6988 | val_acc=0.5000 | val_auc=0.5467
[BASE] Epoch 09/15 | train_loss=0.6878 | val_acc=0.5667 | val_auc=0.5733
[BASE] Epoch 10/15 | train_loss=0.7018 | val_acc=0.5333 | val_auc=0.6267
[BASE] Epoch 11/15 | train_loss=0.6923 | val_acc=0.4667 | val_auc=0.6000
[BASE] Epoch 12/15 | train_loss=0.6865 | val_acc=0.5000 | val_auc=0.5422
[BASE] Epoch 13/15 | train_loss=0.6715 | val_acc=0.6333 | v